# UlamAI Prover Tutorial with Examples\n\nThis notebook is a detailed quickstart for new users.\n\nIt covers:\n1. what to run first\n2. provider setup (Codex recommended)\n3. prove to `.tex` (infinitely many primes)\n4. formalize olympiad `.tex` inputs, including `pol25.tex` (full informal proof)\n5. artifacts and resume basics\n

## 0) First Run (Recommended Order)\n\n1. Setup repo + install package.\n2. Run smoke prove command to verify CLI wiring.\n3. Run `.tex` proving route.\n4. Run formalization on statement-only and full-proof olympiad files.

In [ ]:
%%bash\nset -e\nif [ ! -d ulamai ]; then\n  git clone https://github.com/ulamai/ulamai.git\nfi\ncd ulamai\npython3 -m pip install -q -e .\npython3 -m ulam --help >/dev/null\necho "Setup complete."\n

## 1) Provider Setup\n\nRecommended provider: `codex_cli`.\n\n- Local auth flow: `ulam auth codex`\n- TUI path: `ulam` -> `Configure LLM` -> `Codex`\n\nFor this notebook to run without external keys, we configure `mock` below.\nReplace with real provider later for production-quality outputs.

In [ ]:
%%bash\nset -e\ncd ulamai\nmkdir -p .ulam\ncat > .ulam/config.json <<'JSON'\n{\n  "llm_provider": "mock",\n  "prove": {\n    "mode": "llm",\n    "output_format": "lean",\n    "solver": "script",\n    "autop": true,\n    "k": 1,\n    "tex_out_dir": "proofs",\n    "tex_rounds": 2,\n    "tex_judge_repairs": 1,\n    "tex_worker_drafts": 1,\n    "tex_replan_passes": 1,\n    "tex_artifacts_dir": "runs/prove_tex",\n    "llm_rounds": 2,\n    "llm_cycle_patience": 1,\n    "llm_allow_helper_lemmas": true,\n    "llm_edit_scope": "full",\n    "llm_typecheck_backend": "cli",\n    "search_lean_backend": "dojo",\n    "lemma_max": 60,\n    "lemma_depth": 60,\n    "allow_axioms": true,\n    "typecheck_timeout_s": 30,\n    "retriever_k": 8,\n    "retriever_source": "local",\n    "retriever_build": "auto",\n    "retriever_index": ""\n  },\n  "formalize": {\n    "proof_backend": "llm",\n    "lean_backend": "dojo",\n    "max_rounds": 2,\n    "max_repairs": 2,\n    "max_equivalence_repairs": 1,\n    "max_proof_rounds": 1,\n    "proof_repair": 1,\n    "typecheck_timeout_s": 30,\n    "llm_check": false,\n    "llm_check_timing": "end",\n    "llm_check_repairs": 0\n  },\n  "segmentation": {\n    "chunk_words": 1000\n  },\n  "lean": {\n    "project": "",\n    "imports": [],\n    "dojo_timeout_s": 120\n  },\n  "policy": {\n    "proof_profile": "normal"\n  }\n}\nJSON\necho "Configured mock tutorial profile."\n

In [ ]:
%%bash\ncd ulamai\nls -1 examples\n

## 2) Smoke Check (Lean file)

In [ ]:
%%bash\nset -e\ncd ulamai\npython3 -m ulam prove examples/Smoke.lean \\n  --theorem irrational_sqrt_two_smoke \\n  --llm mock --lean mock\n

## 3) Prove to `.tex` (Infinitely Many Primes)\n\nInput statement file: `examples/ProveTexPrimes.txt`.\n\nFor real runs, replace `--llm mock` with `--llm codex_cli`.

In [ ]:
%%bash\nset -e\ncd ulamai\npython3 -m ulam prove \\n  --theorem infinitely_many_primes \\n  --output-format tex \\n  --statement "$(cat examples/ProveTexPrimes.txt)" \\n  --llm mock \\n  --tex-rounds 2 \\n  --tex-worker-drafts 1 \\n  --tex-judge-repairs 1 \\n  --tex-replan-passes 1 \\n  --tex-artifacts-dir runs/prove_tex\n

In [ ]:
%%bash\ncd ulamai\necho "Proof outputs:"\nls -lah proofs || true\necho\necho "TeX artifacts:"\nls -lah runs/prove_tex || true\n

## 4) Formalize the Olympiad Problem\n\nTwo inputs represent the same theorem:\n- `examples/FormalizePolishOlympiad.tex`: statement-only\n- `examples/pol25.tex`: full informal proof (recommended)

In [ ]:
%%bash\nset -e\ncd ulamai\npython3 -m ulam formalize examples/FormalizePolishOlympiad.tex \\n  --out examples/FormalizePolishOlympiad.lean \\n  --proof-backend llm \\n  --lean-backend dojo \\n  --max-rounds 2 \\n  --max-proof-rounds 1 \\n  --no-equivalence \\n  --no-llm-check \\n  --artifacts-dir runs/formalize_olympiad_stmt\n

In [ ]:
%%bash\nset -e\ncd ulamai\npython3 -m ulam formalize examples/pol25.tex \\n  --out examples/pol25.lean \\n  --proof-backend llm \\n  --lean-backend dojo \\n  --max-rounds 2 \\n  --max-proof-rounds 1 \\n  --no-equivalence \\n  --no-llm-check \\n  --artifacts-dir runs/formalize_olympiad_full\n

In [ ]:
%%bash\ncd ulamai\necho "Statement-only output preview:"\nsed -n '1,120p' examples/FormalizePolishOlympiad.lean || true\necho\necho "Full-proof output preview:"\nsed -n '1,120p' examples/pol25.lean || true\necho\necho "Formalize artifacts:"\nls -lah runs | grep formalize_ || true\n

## 5) Move from Mock to Real LLM Runs\n\nRecommended production settings:\n- provider: `codex_cli`\n- prove mode: `llm`\n- informal-first route: `--output-format tex`\n\nLocal TUI path:\n1. `ulam`\n2. `Configure LLM` -> `Codex`\n3. `Settings` -> `Prover settings` -> set mode to `llm`\n\nThen rerun sections 3 and 4 with real provider settings for higher-quality outputs.